# ToolRL Reproduction Notebook

Provisions a Chameleon Cloud [cloud computing platform for research] GPU instance, sets up Docker [software container system], and runs one training case end-to-end.

**Target:** GRPO [Group Relative Policy Optimization -- RL algorithm that doesn't need a separate critic model] Cold Start [trained straight from the base model, no supervised fine-tuning first], Qwen2.5-1.5B [1.5 billion parameter language model by Alibaba]

**Paper result (API-Bank [the paper's own 597-question tool-call benchmark]):** 63.15% | **Our result:** 58.96%

**Wall time:** ~3.5 hours on 4x H100 [NVIDIA GPU]

**Before running:**
- Chameleon Cloud account with an active allocation at kvm.tacc.chameleoncloud.org
- SSH keypair [public/private key pair for secure remote login] registered in the KVM@TACC dashboard
- `pip install python-chi`

---

## Table of Contents

| Section | What it does |
|---------|-------------|
| [0. Configuration](#0-configuration) | Set your project name, keypair, GPU flavor |
| [1. Provision Instance](#1-provision-the-instance) | Create lease + launch VM. Cell 1c creates a Cinder volume [persistent disk] |
| [2. Connect via SSH](#2-connect-via-ssh) | Open remote connection to the VM |
| [3. Install Docker](#3-install-docker-and-nvidia-container-toolkit) | Install Docker + NVIDIA container support |
| [4. Clone Repo & Volumes](#4-clone-repo-and-create-volumes) | Download code, create Docker named volumes [persistent storage buckets] |
| [5. Build Docker Image](#5-build-the-docker-image) | Build the training container (~15-20 min) |
| [6. Start Containers](#6-start-containers) | Launch verl [training container] + mlflow [experiment tracker] |
| [7. Prepare Dataset](#7-prepare-the-dataset) | Convert raw JSON to parquet [columnar data format the trainer needs] |
| [8. Download Base Model](#8-download-the-base-model) | Pull Qwen2.5-1.5B from HuggingFace [model hosting platform] |
| [9. Run Training](#9-run-training) | Run GRPO demo (1 step, ~4-5 min) or full 15-epoch reproduction |
| [10. Evaluate on API-Bank](#10-evaluate-on-api-bank) | Run 597-question benchmark, get accuracy score |
| [11. Check Disk & Checkpoints](#11-check-disk--clean-old-checkpoints) | Monitor disk, kill jobs, clean old checkpoints [saved model weights] |
| [12. Cleanup](#12-cleanup) | Check lease, stop containers, delete server/lease/volume |
| [(Optional) Resume from Volume](#optional-resume-from-existing-cinder-volume) | Boot new VM from existing Cinder volume -- skips all setup |

## 0. Configuration

In [ ]:
SITE         = "KVM@TACC"
PROJECT_NAME = "CHI-XXXXXX"       # your allocation number, e.g. CHI-231138
KEYPAIR_NAME = "my-chameleon-key" # as it appears in the dashboard at kvm.tacc.chameleoncloud.org
SSH_KEY_PATH = "~/.ssh/id_rsa"    # path to your private key inside JupyterHub
LEASE_HOURS  = 8
REPO_URL     = "https://github.com/Mario928/toolrl-verl-reproduction"

# --- GPU flavor: pick one based on what's available in your allocation ---
# FLAVOR = "gpu.h100.4x"   # 4x H100 NVL 94GB -- faster (~3.5h), set N_GPUS=4 below
# FLAVOR = "g1.h100.pci.1" # 1x H100 PCIe     -- slower (~7h),   set N_GPUS=1 below
FLAVOR = "gpu.h100.4x"    # change to g1.h100.pci.1 if 4x is unavailable

# --- GPU count: must match FLAVOR above ---
# N_GPUS = 1   # for g1.h100.pci.1
# N_GPUS = 4   # for gpu.h100.4x
N_GPUS = 4


## 1. Provision the Instance

**Using an existing active KVM@TACC lease:**
Cell 1a (lease creation) is commented out. Just run cell 1b - it launches the instance directly using the `FLAVOR` set in the config cell above.

If you don't have an active lease yet, uncomment cell 1a and run it first.

In [ ]:
# --- 1a: Create lease (SKIP if you already have an active KVM@TACC lease) ---
# Uncomment and run this cell only if you need a new lease.

# import chi, chi.lease, chi.server, chi.jupyterhub, datetime
#
# chi.use_site(SITE)
# chi.set("project_name", PROJECT_NAME)
# if chi.jupyterhub.is_jupyterhub_env():
#     chi.set("auth_type", "v3oidcaccesstoken")
#
# lease_obj = chi.lease.Lease(
#     "toolrl-reproduce",
#     duration=datetime.timedelta(hours=LEASE_HOURS),
# )
# lease_obj.add_flavor_reservation(name=FLAVOR, amount=1)
# lease_obj.submit(idempotent=True)
# print("Lease active:", lease_obj.id)

### 1c. (Optional) Create a Cinder Block Volume

Skip this if you already have a volume from a previous run (check KVM@TACC dashboard > Volumes).

A Cinder volume gives you persistent large disk (512GB+) that survives instance deletion. Without it, the VM gets ~40GB which is not enough for the Docker image + model weights (~60-70GB needed).

After creating it, use the boot-from-volume cell at the bottom of this notebook instead of cell 1b.

In [ ]:
# --- 1c: Create a Cinder block volume (SKIP if you already have one) ---
# Creates a 512GB persistent volume at KVM@TACC.
# Only needs to be done once -- the volume persists across reservations.

# CINDER_VOLUME_NAME = "toolrl-storage"   # any name you want
# CINDER_VOLUME_SIZE = 512                # GB -- 512 is enough for image + models + checkpoints

# import chi, requests
# chi.use_site(SITE)
# chi.set("project_name", PROJECT_NAME)
# if chi.jupyterhub.is_jupyterhub_env():
#     chi.set("auth_type", "v3oidcaccesstoken")
#
# session  = chi.session()
# token    = session.get_token()
# cinder_url = "https://kvm.tacc.chameleoncloud.org:8776/v3/volumes"
#
# payload = {
#     "volume": {
#         "name": CINDER_VOLUME_NAME,
#         "size": CINDER_VOLUME_SIZE,
#         "volume_type": "ceph",         # standard type at KVM@TACC
#         "availability_zone": "nova",
#     }
# }
# r = requests.post(cinder_url, headers={"X-Auth-Token": token, "Content-Type": "application/json"}, json=payload, timeout=30)
# vol = r.json()["volume"]
# print("Volume ID:", vol["id"])
# print("Status:", vol["status"])  # will show 'creating' then 'available'
# # Save this Volume ID -- you'll need it for the boot-from-volume cell at the bottom

In [ ]:
# --- 1b: Launch instance ---
# Uses FLAVOR from the config cell above.
# If you ran cell 1a, you can also use: lease_obj.get_reserved_flavors()[0].name
import chi, chi.server, chi.jupyterhub

chi.use_site(SITE)
chi.set("project_name", PROJECT_NAME)
if chi.jupyterhub.is_jupyterhub_env():
    chi.set("auth_type", "v3oidcaccesstoken")

server = chi.server.create_server(
    "toolrl-node",
    flavor_name=FLAVOR,
    image_name="CC-Ubuntu22.04",
    key_name=KEYPAIR_NAME,
)
chi.server.wait_for_active(server.id)
floating_ip = chi.server.associate_floating_ip(server.id)
print("Floating IP:", floating_ip)

## 2. Connect via SSH

In [ ]:
import chi.ssh, os

node = chi.ssh.Remote(floating_ip, username="cc", key_filename=os.path.expanduser(SSH_KEY_PATH))
stdout, _ = node.execute("uname -a")
print(stdout)

## 3. Install Docker and nvidia-container-toolkit

In [ ]:
node.execute("sudo apt-get update -q")
node.execute("sudo apt-get install -y -q ca-certificates curl gnupg lsb-release")
node.execute("curl -fsSL https://download.docker.com/linux/ubuntu/gpg | sudo gpg --dearmor -o /usr/share/keyrings/docker-archive-keyring.gpg")
node.execute('echo "deb [arch=$(dpkg --print-architecture) signed-by=/usr/share/keyrings/docker-archive-keyring.gpg] https://download.docker.com/linux/ubuntu $(lsb_release -cs) stable" | sudo tee /etc/apt/sources.list.d/docker.list > /dev/null')
node.execute("sudo apt-get update -q")
node.execute("sudo apt-get install -y -q docker-ce docker-ce-cli containerd.io docker-compose-plugin")
node.execute("sudo usermod -aG docker cc")
print("Docker installed.")

In [ ]:
node.execute("curl -fsSL https://nvidia.github.io/libnvidia-container/gpgkey | sudo gpg --dearmor -o /usr/share/keyrings/nvidia-container-toolkit-keyring.gpg")
node.execute("curl -s -L https://nvidia.github.io/libnvidia-container/stable/deb/nvidia-container-toolkit.list | sed 's#deb https://#deb [signed-by=/usr/share/keyrings/nvidia-container-toolkit-keyring.gpg] https://#g' | sudo tee /etc/apt/sources.list.d/nvidia-container-toolkit.list")
node.execute("sudo apt-get update -q")
node.execute("sudo apt-get install -y -q nvidia-container-toolkit")
node.execute("sudo nvidia-ctk runtime configure --runtime=docker")
node.execute("sudo systemctl restart docker")
print("nvidia-container-toolkit installed.")

In [ ]:
stdout, _ = node.execute("sudo docker run --rm --gpus all nvidia/cuda:12.1.0-base-ubuntu22.04 nvidia-smi --query-gpu=name,memory.total --format=csv,noheader")
print(stdout)

## 4. Clone Repo and Create Volumes

In [ ]:
node.execute(f"git clone {REPO_URL} /home/cc/toolrl")
for vol in ["toolrl_models", "toolrl_hf_cache", "toolrl_mlflow_data", "toolrl_datasets"]:
    node.execute(f"sudo docker volume create {vol}")
print("Done.")

## 5. Build the Docker Image

Takes ~15-20 minutes the first time.

In [ ]:
stdout, _ = node.execute("cd /home/cc/toolrl && sudo docker build -t toolrl-verl:latest . 2>&1 | tail -20")
print(stdout)

## 6. Start Containers

In [ ]:
node.execute("cd /home/cc/toolrl && sudo docker compose up -d")
stdout, _ = node.execute("sudo docker ps --format 'table {{.Names}}\t{{.Status}}'")
print(stdout)
print(f"MLflow UI [experiment tracker -- tracks loss, reward, and training metrics]: http://{floating_ip}:5000")

## 7. Prepare the Dataset

Converts the raw JSON files already in the repo into the parquet format the trainer expects. Must run before training or the trainer crashes at step 1 with `KeyError: 'reward_model'`.

In [ ]:
node.execute("sudo docker exec verl bash -c 'cd /workspace && python dataset/rlla_4k_raw/rlla.py'")
print("Dataset prepared.")

In [ ]:
node.execute(
    "sudo docker exec verl python3 -c "
    "'from huggingface_hub import snapshot_download; "
    "snapshot_download(\"Qwen/Qwen2.5-1.5B-Instruct\")'"
)
print("Model downloaded.")

## 9. Run Training

**Demo mode (active below):** 1 epoch, small batch — completes in ~10-15 min on 1x H100. Shows the full training loop working end-to-end.

**Full reproduction run:** 15 epochs, batch=512 — takes ~3.5h on 4x H100 / ~7h on 1x H100. Reaches the paper's checkpoint at `global_step_90` with 58.96% API-Bank accuracy. Swap in the commented config below to run it.

To run a different reward variant, set the reward flags (`COARSEREWARD=1`, `REFINEDREWARD=1`, etc.) before the training command.

In [ ]:
# --- Run demo training: 1 step, batch=64, ~10-15 min on 1x H100 ---
node.execute("""sudo docker exec -d -e EXPERIMENT_NAME=grpo_qwen_1.5b_demo verl bash -c 'cd /workspace && CUDA_VISIBLE_DEVICES=0 VLLM_ATTENTION_BACKEND=XFORMERS python3 -m verl.trainer.main_ppo algorithm.adv_estimator=grpo data.train_files=./dataset/rlla_4k/train_64.parquet data.val_files=./dataset/rlla_4k/test.parquet data.train_batch_size=64 data.val_batch_size=32 data.max_prompt_length=2048 data.max_response_length=1024 actor_rollout_ref.model.path=Qwen/Qwen2.5-1.5B-Instruct actor_rollout_ref.actor.optim.lr=1e-6 actor_rollout_ref.model.use_remove_padding=True actor_rollout_ref.actor.ppo_mini_batch_size=64 actor_rollout_ref.actor.use_dynamic_bsz=True actor_rollout_ref.actor.use_kl_loss=False actor_rollout_ref.actor.kl_loss_coef=0.001 actor_rollout_ref.actor.kl_loss_type=low_var_kl actor_rollout_ref.model.enable_gradient_checkpointing=True actor_rollout_ref.actor.fsdp_config.param_offload=False actor_rollout_ref.actor.fsdp_config.grad_offload=False actor_rollout_ref.actor.fsdp_config.optimizer_offload=False actor_rollout_ref.rollout.tensor_model_parallel_size=1 actor_rollout_ref.rollout.name=vllm actor_rollout_ref.rollout.gpu_memory_utilization=0.6 actor_rollout_ref.rollout.n=4 actor_rollout_ref.ref.fsdp_config.param_offload=True algorithm.kl_ctrl.kl_coef=0.001 trainer.critic_warmup=0 trainer.logger=[console,mlflow] trainer.project_name=toolrl trainer.default_local_dir=/app/models/toolrl-grpo-cold-qwen-1.5b-demo trainer.experiment_name=grpo_qwen_1.5b_demo trainer.n_gpus_per_node=1 trainer.nnodes=1 trainer.save_freq=1 trainer.test_freq=1 trainer.total_epochs=1 > /tmp/train.log 2>&1'""")
print(f"Training launched. MLflow: http://{floating_ip}:5000")

# Tail log until step:1 appears (run this in a loop or re-run manually)
import time
for _ in range(40):
    stdout, _ = node.execute("sudo docker exec verl tail -3 /tmp/train.log 2>/dev/null || echo 'starting...'")
    print(stdout.strip())
    if 'step:1' in stdout or 'Final validation' in stdout:
        print("\nTraining complete!")
        break
    time.sleep(20)

### Inspect: Dataset, GPU, Model Output

Run any of these cells during or after training to answer professor questions live.

In [ ]:
# --- Show a sample from the training dataset [the tool-call questions the model learns from] ---
stdout, _ = node.execute("""
sudo docker exec verl python3 -c "
import pandas as pd, json
df = pd.read_parquet('./dataset/rlla_4k/train.parquet')
print(f'Train size: {len(df)} rows | Columns: {list(df.columns)}')
print()
row = df.iloc[0]
# prompt is a list of chat messages
msgs = row['prompt'] if isinstance(row['prompt'], list) else json.loads(row['prompt'])
for m in msgs:
    print(f'[{m[\"role\"].upper()}]')
    print(m['content'][:300])
    print()
"
""")
print(stdout)

In [ ]:
# --- GPU utilization [how hard the GPU is working] during training ---
stdout, _ = node.execute("sudo docker exec verl nvidia-smi --query-gpu=name,utilization.gpu,memory.used,memory.total,temperature.gpu --format=csv,noheader")
print(stdout)

In [ ]:
# --- Show a raw model output sample [what the model actually generates for a tool-call question] ---
# Run after training completes. Uses the demo checkpoint.
stdout, _ = node.execute("""
sudo docker exec verl python3 -c "
from vllm import LLM, SamplingParams
import pandas as pd

llm = LLM(model='/app/models/toolrl-grpo-cold-qwen-1.5b-demo/actor/global_step_1', gpu_memory_utilization=0.6)
params = SamplingParams(temperature=0.0, max_tokens=256)

df = pd.read_parquet('./dataset/rlla_4k/test.parquet')
msgs = df.iloc[0]['prompt']
prompt = '\n'.join(f'[{m[\"role\"].upper()}] {m[\"content\"]}' for m in msgs) + '\n[ASSISTANT]'
out = llm.generate([prompt], params)[0].outputs[0].text
print('=== PROMPT (first test question) ===')
print(prompt[:500])
print()
print('=== MODEL OUTPUT ===')
print(out)
"
""")
print(stdout)

In [ ]:
# --- Show the reward function [the code that scores whether the model's tool call was correct] ---
# stdout, _ = node.execute("sudo docker exec verl cat /workspace/verl/utils/reward_score/rlla.py")
# print(stdout)

# HOW THE REWARD FUNCTION WORKS:
# --------------------------------
# 1. The model generates a response to a tool-call question.
#    Example question: "What is the weather in New York?"
#    Expected output:  <tool_call>GetWeather(location="New York")</tool_call>
#
# 2. FORMAT CHECK [did the model wrap its answer in <tool_call> tags?]
#    - Correct format  -> +1
#    - Missing tags    ->  0  (model didn't learn the output structure yet)
#
# 3. CORRECTNESS CHECK [did the model call the right API with the right parameters?]
#    - Exact match [API name + all parameters correct] -> +3
#    - Wrong API name                                  -> -3
#    - Right API, wrong parameters                     -> -3 (default) or partial credit if COARSEREWARD=1
#
# 4. FINAL SCORE = format score + correctness score
#    Range: -3 (totally wrong) to +4 (correct format AND correct answer)
#
# 5. GRPO [Group Relative Policy Optimization] uses these scores to update the model:
#    - Generates 4 responses [rollout.n=4] for each question
#    - Compares scores within the group: responses better than average get positive advantage [pushed up]
#    - Responses worse than average get negative advantage [pushed down]
#    - No separate critic model needed [unlike PPO] -- the group comparison IS the baseline

## 12. Cleanup

Run in order. Uncomment and run one cell at a time.

**Important:** Deleting the server does NOT delete the Cinder volume [the persistent disk with all your data]. The volume stays until you explicitly delete it. So your checkpoints and MLflow data are safe even after server deletion.

In [ ]:
# --- Check reservation status [is your GPU lease still active or expired?] ---
import chi, chi.lease
chi.use_site(SITE)
chi.set("project_name", PROJECT_NAME)
if chi.jupyterhub.is_jupyterhub_env():
    chi.set("auth_type", "v3oidcaccesstoken")

for l in chi.lease.list_leases():
    print(f"Lease: {l['name']} | Status: {l['status']} | Ends: {l['end_date']}")
    for r in l.get("reservations", []):
        print(f"  reservation: {r['id']} | type: {r.get('resource_type')} | status: {r.get('status')}")

In [ ]:
# --- Stop containers (run before deleting server) ---
# node.execute("cd /home/cc/toolrl && sudo docker compose down")
# print("Containers stopped.")

In [ ]:
# --- Delete the server [shuts down the VM, frees the GPU reservation] ---
# The Cinder volume [your persistent disk] is NOT deleted -- data stays safe.
# chi.server.delete_server(server.id)
# print("Server deleted.")

In [ ]:
# --- Delete the lease [releases the GPU reservation back to the pool] ---
# Only do this when fully done -- once deleted you need a new lease to get GPUs again.
# lease_obj.delete()
# print("Lease deleted.")

In [ ]:
# --- Delete the Cinder volume [WARNING: permanently deletes all checkpoints and MLflow data] ---
# Only run this if you are completely done and don't need the data anymore.
# VOLUME_ID = "067b7d0d-XXXX-XXXX-XXXX-XXXXXXXXXXXX"  # your volume ID
# import requests
# session = chi.session()
# token = session.get_token()
# r = requests.delete(
#     f"https://kvm.tacc.chameleoncloud.org:8776/v3/volumes/{VOLUME_ID}",
#     headers={"X-Auth-Token": token},
#     timeout=30
# )
# print("Deleted" if r.status_code == 202 else f"Error: {r.status_code} {r.text}")

In [ ]:
# Checkpoint to evaluate -- demo run saves to global_step_1
CHECKPOINT = "/app/models/toolrl-grpo-cold-qwen-1.5b-demo/actor/global_step_1"

# Step 1: Generate model outputs on the 597 API-Bank questions (~2-3 min)
print("Generating outputs...")
stdout, _ = node.execute(f"""
sudo docker exec verl bash -c '
    cd /workspace/benchmarks/API-Bank &&
    python3 generate_batch.py --model_paths {CHECKPOINT}
'
""")
print(stdout)

# Step 2: Score the outputs
print("Scoring...")
stdout, _ = node.execute(f"""
sudo docker exec verl bash -c '
    cd /workspace/benchmarks/API-Bank &&
    python3 evaluate.py --model_paths {CHECKPOINT}
'
""")
print(stdout)

## 11. Check Disk & Clean Old Checkpoints

Run before starting a new training run to avoid disk full errors.
Each 1.5B checkpoint is ~6-7 GB. The demo run saves one per step.

In [ ]:
# --- Check disk and existing checkpoints ---
stdout, _ = node.execute("df -h / | tail -1")
print("Disk:", stdout.strip())
stdout, _ = node.execute("sudo docker exec verl bash -c 'du -sh /app/models/toolrl-grpo-cold-qwen-1.5b-demo/actor/*/ 2>/dev/null | sort -V || echo empty'")
print("Checkpoints:\n", stdout)

# --- Kill any running training job (run before starting a new one) ---
# node.execute("sudo docker exec verl pkill -f 'verl.trainer.main_ppo'")
# print("Killed.")

# --- Delete ALL checkpoints (nuke everything, max disk freed) ---
# node.execute("sudo docker exec verl bash -c 'rm -rf /app/models/toolrl-grpo-cold-qwen-1.5b-demo && rm -f /tmp/train.log'")
# print("Cleaned all.")

# --- Delete specific checkpoints (keep only the ones you want) ---
# KEEP = {"global_step_1"}   # <-- set which steps to keep
# stdout, _ = node.execute("sudo docker exec verl bash -c 'ls /app/models/toolrl-grpo-cold-qwen-1.5b-demo/actor/'")
# all_steps = set(stdout.strip().split())
# to_delete = all_steps - KEEP
# for step in sorted(to_delete):
#     node.execute(f"sudo docker exec verl rm -rf /app/models/toolrl-grpo-cold-qwen-1.5b-demo/actor/{step}")
#     print(f"Deleted {step}")
# print(f"Kept: {KEEP}")

# --- Check training log (run anytime to see progress) ---
# stdout, _ = node.execute("sudo docker exec verl tail -20 /tmp/train.log 2>/dev/null || echo 'No log yet'")
# print(stdout)

# --- Check MLflow runs ---
# stdout, _ = node.execute("curl -s 'http://localhost:5000/api/2.0/mlflow/runs/search' -d '{\"experiment_ids\":[\"949894919383340008\"],\"max_results\":3,\"order_by\":[\"attribute.start_time DESC\"]}' -H 'Content-Type: application/json' | python3 -c 'import sys,json; [print(r[\"info\"][\"run_name\"],\"|\",r[\"info\"][\"status\"]) for r in json.load(sys.stdin)[\"runs\"]]'")
# print(stdout)

## 11. Cleanup

Run in order. Uncomment and run one cell at a time.

In [ ]:
# Stop containers
# node.execute("cd /home/cc/toolrl && sudo docker compose down")

In [ ]:
# Delete the server
# chi.server.delete_server(server.id)

In [ ]:
# Delete the lease
# lease_obj.delete()
# print("Lease deleted.")

## (Optional) Resume from Existing Cinder Volume

Use this instead of Sections 1-8 if you already ran a full setup before and your Cinder volume still exists.

**What this does:** boots a new VM directly from your existing 512GB Cinder volume. Docker, the image, containers, checkpoints, HuggingFace cache, and MLflow data are all already there -- no reinstall needed.

**When to use:**
- Your previous reservation expired but you kept the Cinder volume
- You want to avoid the ~20 min Docker build + model download

**Requirements:**
- You know your Cinder volume ID (e.g. `067b7d0d-...`) -- find it in the KVM@TACC dashboard under Volumes
- You have an active lease with a GPU flavor reserved
- Run the config cell (Section 0) first

In [ ]:
# --- Resume from existing Cinder volume (replaces Sections 1-8) ---
# Fill in your volume ID and reservation ID, then uncomment and run.

# VOLUME_ID      = "067b7d0d-XXXX-XXXX-XXXX-XXXXXXXXXXXX"  # your Cinder volume ID -- KVM@TACC dashboard > Volumes
# NETWORK_ID     = "50073c73-5817-49c3-8e3a-69b8c357e158"  # sharednet1 -- same for all KVM@TACC projects

# --- Step 1: Find your reservation ID from your active lease ---
# import chi, chi.lease
# chi.use_site(SITE)
# chi.set("project_name", PROJECT_NAME)
# if chi.jupyterhub.is_jupyterhub_env():
#     chi.set("auth_type", "v3oidcaccesstoken")
# for l in chi.lease.list_leases():
#     print(l["name"], "| lease:", l["id"])
#     for r in l.get("reservations", []):
#         print("  reservation:", r["id"], "|", r.get("resource_type"))
# # Copy the reservation ID for the flavor/physical reservation and set it below:
# RESERVATION_ID = "XXXXXXXX-XXXX-XXXX-XXXX-XXXXXXXXXXXX"

# --- Step 2: Boot new VM from the existing volume ---
# import requests
# session  = chi.session()
# token    = session.get_token()
# nova_url = "https://kvm.tacc.chameleoncloud.org:8774/v2.1/servers"
#
# payload = {
#     "server": {
#         "name": "toolrl-node",
#         "flavorRef": f"reservation:{RESERVATION_ID}",
#         "key_name": KEYPAIR_NAME,
#         "networks": [{"uuid": NETWORK_ID}],
#         "block_device_mapping_v2": [{
#             "boot_index": 0,
#             "uuid": VOLUME_ID,
#             "source_type": "volume",
#             "destination_type": "volume",
#             "delete_on_termination": False   # keeps volume safe if server is deleted
#         }]
#     }
# }
# r = requests.post(nova_url, headers={"X-Auth-Token": token, "Content-Type": "application/json"}, json=payload, timeout=30)
# server_id = r.json()["server"]["id"]
# print("Server ID:", server_id)
# chi.server.wait_for_active(server_id)
# floating_ip = chi.server.associate_floating_ip(server_id)
# print("Floating IP:", floating_ip)

# --- Step 3: SSH in and restart containers (Docker image + data already on volume) ---
# import chi.ssh, os
# node = chi.ssh.Remote(floating_ip, username="cc", key_filename=os.path.expanduser(SSH_KEY_PATH))
# node.execute("cd /home/cc/toolrl && sudo docker compose up -d")
# stdout, _ = node.execute("sudo docker ps --format 'table {{.Names}}\t{{.Status}}'")
# print(stdout)
# # Done -- skip to Section 9 (training) or Section 10 (eval)